## Create instructions for all data sets

In [ ]:
STORY_INSTRUCTIONS = [
    "Tell me a story about <|insert|>",
    "Can you tell me a story about <|insert|>?",
    "Write a short story about <|insert|>",
    "Make up a story about <|insert|>",
    "I want to hear a story about <|insert|>",
    "Create a fun story about <|insert|>",
    "Please tell me a story about <|insert|>",
    "Invent a story about <|insert|>",
    "Tell a bedtime story about <|insert|>",
    "Can you make a story about <|insert|>?",
    
    "Wanna hear a fun story? Tell me about <|insert|>",
    "Hey, tell me a story about <|insert|>",
    "Could you write a story about <|insert|>?",
    "Give me a story about <|insert|>",
    "Do you know a story about <|insert|>?",
    
    "Tell me a simple story about <|insert|>",
    "Tell me a children's story about <|insert|>",
    "Tell me a happy story about <|insert|>",
    "Tell me a funny story about <|insert|>",
    "Tell me an interesting story about <|insert|>",
    
    "Write a creative story about <|insert|>",
    "Write a nice story about <|insert|>",
    "Write a fun little story about <|insert|>",
    "Write a short and simple story about <|insert|>",
    
    "Imagine a story about <|insert|> and tell it to me",
    "Can you imagine a story about <|insert|>?",
    "Make up a creative story about <|insert|>",
    
    "Let’s hear a story about <|insert|>",
    "Tell me something about <|insert|> in story form",
    "Turn <|insert|> into a story",
    
    "Tell me a story involving <|insert|>",
    "Create a story where <|insert|> is important",
    "Write a story that includes <|insert|>",
    
    "Can you tell me a bedtime story about <|insert|>?",
    "Tell me a relaxing story about <|insert|>",
    
    "Tell me a story with <|insert|> in it",
    "Write a story where <|insert|> appears",
]

WIKI_INSTRUCTIONS = [
    "Explain <|insert|> in simple terms.",
    "Write a short paragraph about <|insert|>.",
    "What is <|insert|>?",
    "Give an overview of <|insert|>.",
    "Describe how <|insert|> works.",
    "Why is <|insert|> important?",
    "Explain the concept of <|insert|>.",
    "Summarize <|insert|>.",
    "Teach me about <|insert|> like I'm a beginner.",
    "What are the key facts about <|insert|>?",
    "Write a simple explanation of <|insert|>.",
    "Explain <|insert|> with an example.",
]

FACTUAL_INSTRUCTIONS = [
    "Explain <|insert|> in simple terms.",
    "What is <|insert|>?",
    "Describe <|insert|>.",
    "Give an overview of <|insert|>.",
    "Why is <|insert|> important?",
    "Explain <|insert|> like I'm a beginner.",
]

SUMMARY_INSTRUCTIONS = [
    "Summarize this text.",
    "Give a short summary of this text.",
    "What is the main idea of this text?",
    "Briefly explain what this text is about.",
]

OPINION_INSTRUCTIONS = [
    "What is the main opinion in this text?",
    "What argument is being made?",
    "Summarize the author's point of view.",
]

In [7]:
import numpy as np

from NoTorchAI.NLP.Rake import Rake
from NoTorchAI.NLP.KeyBERT import KeyBERT

MAX_BUFFER = 500_000
BUFFER = ''

In [8]:
def get_best_key_words_len_2_4(key_words: dict) -> str:
        key_words_2_to_4 = {
            k: v for k, v in key_words.items()
            if 2 <= len(k.split()) <= 4
        }
        if key_words_2_to_4:
            best_key, _ = max(key_words_2_to_4.items(), key=lambda kv: kv[1])
        else:
            best_key, _ = max(key_words.items(), key=lambda kv: kv[1])

        return best_key

In [ ]:
def stories_with_instructions():
    global BUFFER, MAX_BUFFER
    
    #rake = Rake()
    bert = KeyBERT("env/tokenizer.json") 
    # rake.add_stop_word(["one", "day", "said", "hi", 
    #                     "now", "will", "says", "away", 
    #                     "always", "oh", "around"])

    with open("env/children_stories.txt", "r", encoding="utf-8") as f:
        stop = "<|endoftext|>"
        story = "" 

        for line_id, line in enumerate(f):
            if line_id == 5_000_000:
                break

            if stop in line:

                key_sentences = bert.get_key_words(story, 1, 1)
                if not key_sentences:
                    continue

                best_key = max(key_sentences.items(), key=lambda kv: kv[1])[0]
                best_key = best_key.strip()

                #promt_template = np.random.choice(STORY_INSTRUCTIONS)  # returns scalar string
                promt = best_key #promt_template.replace("<|insert|>", best_key)
                first_output = f"Here is a story based on this sentence: {promt}.\n"
                BUFFER += "<|user|>: " + promt + "\n<|assistant|>: " + first_output + story + stop + "\n\n"
                story = ""
                continue

            if len(BUFFER) > MAX_BUFFER:
                with open("env/stories_instructions.txt", "a", encoding="utf-8") as out_file:
                    out_file.write(BUFFER)
                BUFFER = ""

            story += line

In [18]:
stories_with_instructions()

KeyboardInterrupt: 

In [ ]:
from datasets import load_dataset


def wiki_with_instructions():
    global BUFFER, MAX_BUFFER

    from datasets import load_dataset
    import numpy as np

    rake = Rake()
    rake.add_stop_word([
        "used", "using", "also", "known", "called",
        "one", "two", "first", "second",
        "many", "often", "usually"
])

    ds = load_dataset("rahular/simple-wikipedia", split="train")

    for row_id, row in enumerate(ds):
        if row_id == 5_000_000:
            break

        text = row.get("text") or row.get("document") or ""
        text = text.strip()

        if not text or len(text) < 200:  # Artificial length to exclude titels and so on
            continue

        # only first paragraph
        text = text.split("\n")[0]

        key_words = rake.get_key_words(text)
        if not key_words:
            continue

        best_key = get_best_key_words_len_2_4(key_words)
        best_key = best_key.strip()
        if len(best_key.split()) > 6:
            continue

        prompt_template = np.random.choice(WIKI_INSTRUCTIONS)
        prompt = prompt_template.replace("<|insert|>", best_key)

        BUFFER += "<|user|>: " + prompt + "\n"
        BUFFER += "<|assistant|>: " + text + "\n<|endoftext|>\n\n"

        if len(BUFFER) > MAX_BUFFER:
            with open("env/wiki_instructions.txt", "a", encoding="utf-8") as out_file:
                out_file.write(BUFFER)
            BUFFER = ""


In [ ]:
from datasets import load_dataset


def web_with_instructions():
    global BUFFER, MAX_BUFFER

    import numpy as np

    rake = Rake()
    rake.add_stop_word([
        "said", "says", "also", "would", "could", "should",
        "one", "two", "first", "second",
        "many", "much", "very", "really",
        "get", "got", "go", "went",
        "like", "just", "even", "still",
        "know", "think", "people"
    ])

    ds = load_dataset("Skylion007/openwebtext", split="train", streaming=True)

    for row_id, row in enumerate(ds):
        if row_id == 5_000_000:
            break

        text = row.get("text") or ""
        text = text.strip()

        if not text or len(text) < 200:
            continue

        if len(text) > 2000:
            text = text[:2000]

        if text.count("http") > 2:
            continue

        text = text.split("\n")[0]
        lower = text.lower()

        is_announcement = lower.count("we ") > 5 or "we're" in lower
        is_opinion = "i think" in lower or "in my opinion" in lower
        is_factual = " is " in text[:200] and not is_announcement

        # FACTUAL → use RAKE
        if is_factual:
            key_words = rake.get_key_words(text)
            if not key_words:
                continue

            best_key = get_best_key_words_len_2_4(key_words)
            best_key = best_key.strip()

            if len(best_key.split()) > 6:
                continue

            if any(char.isdigit() for char in best_key):
                continue

            if not best_key.lower().startswith(("a ", "an ", "the ")):
                best_key = "the " + best_key

            prompt_template = np.random.choice(FACTUAL_INSTRUCTIONS)
            prompt = prompt_template.replace("<|insert|>", best_key)

        # OPINION
        elif is_opinion:
            prompt = np.random.choice(OPINION_INSTRUCTIONS)

        # ANNOUNCEMENT / BLOG → summarize
        elif is_announcement:
            prompt = np.random.choice(SUMMARY_INSTRUCTIONS)

        # UNKNOWN → skip
        else:
            continue

        BUFFER += "<|user|>: " + prompt + "\n"
        BUFFER += "<|assistant|>: " + text + "\n<|endoftext|>\n\n"

        if len(BUFFER) > MAX_BUFFER:
            with open("env/web_instructions.txt", "a", encoding="utf-8") as out_file:
                out_file.write(BUFFER)
            BUFFER = ""

    if BUFFER:
        with open("env/web_instructions.txt", "a", encoding="utf-8") as out_file:
            out_file.write(BUFFER)
        BUFFER = ""

## Create all new insructions based on 3 datasets

In [ ]:
BUFFER = ""
stories_with_instructions()
BUFFER = ""
wiki_with_instructions()
BUFFER = ""
web_with_instructions()
BUFFER = ""

## Hugging Face option

In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

ds1 = load_dataset("Skylion007/openwebtext")
ds2 = load_dataset("rahular/simple-wikipedia")


def all_texts():
    with open("children_stories.txt", "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield line.strip()

    for row in ds2:
        text = row.get("text") or row.get("document") or ""
        if text.strip():
            yield text

    for i, row in enumerate(ds1):
        if i >= 1_000_000:
            break
        if row["text"].strip():
            yield row["text"]


tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)  # ByteLevel handles punctuation/unicode better than Whitespace

trainer = trainers.BpeTrainer(
    vocab_size=16_000,
    min_frequency=2,
    special_tokens=["<|endofstory|>", "<|user|>", "<|assistant|>"]
)

tokenizer.train_from_iterator(all_texts(), trainer=trainer)  # train_from_iterator instead of train()

tokenizer.save("tokenizer.json")

## Turn .txt to .bin

In [15]:
import numpy as np
from tokenizers import Tokenizer
from datasets import load_dataset


ds1 = load_dataset("Skylion007/openwebtext")["train"]
ds2 = load_dataset("rahular/simple-wikipedia")["train"]

tokenizer = Tokenizer.from_file("env/tokenizer.json")


def clean_text(text):
    text = ' '.join(text.split())
    return text


def encode_and_save(name, text_iter, output_dir="env/encoded", batch_size=10_000):
    import os
    os.makedirs(output_dir, exist_ok=True)
    out_path = f"{output_dir}/{name}.bin"

    count = 0
    batch = []

    with open(out_path, "ab") as f:
        for text in text_iter:
            cleaned = clean_text(text)
            if not cleaned:
                continue
            batch.extend(tokenizer.encode(cleaned).ids)
            count += 1

            if count % batch_size == 0:
                np.array(batch, dtype=np.int32).tofile(f)
                print(f"  [{name}] {count} texts, flushed {len(batch):,} tokens")
                batch = []

        # flush remainder
        if batch:
            np.array(batch, dtype=np.int32).tofile(f)

    total = os.path.getsize(out_path) // 4  # int32 = 4 bytes
    print(f"  [{name}] Done: {count} texts, {total:,} tokens → {out_path}")
    return total


def texts_from_file(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield line.strip()


def texts_from_dataset(ds, keys=("text", "document", "passage"), limit=None):
    for i, row in enumerate(ds):
        if limit and i >= limit:
            break
        text = ""
        for k in keys:
            text = row.get(k, "") if isinstance(row, dict) else getattr(row, k, "")
            if text and text.strip():
                break
        if text.strip():
            yield text


stats = {}

print("Encoding children_stories...")
stats["children_stories"] = encode_and_save(
    "children_stories",
    texts_from_file("env/children_stories.txt")
)

print("Encoding stories_instructions...")
stats["stories_instructions"] = encode_and_save(
    "stories_instructions",
    texts_from_file("env/stories_instructions.txt")
)

print("Encoding simple_wikipedia...")
stats["simple_wikipedia"] = encode_and_save(
    "simple_wikipedia",
    texts_from_dataset(ds2, keys=("text",))
)

print("Encoding wiki_instructions...")
stats["wiki_instructions"] = encode_and_save(
    "wiki_instructions",
    texts_from_file("env/wiki_instructions.txt")
)

print("Encoding web...")
stats["web"] = encode_and_save(
    "web",
    texts_from_dataset(ds1, keys=("text",), limit=200_000)
)

print("Encoding web_instructions...")
stats["web_instructions"] = encode_and_save(
    "web_instructions",
    texts_from_file("env/web_instructions.txt")
)

print("\n=== Token counts per dataset ===")
total = sum(stats.values())
for name, count in stats.items():
    print(f"  {name:30s} {count:>12,} tokens  ({100*count/total:.1f}%)")
print(f"  {'TOTAL':30s} {total:>12,} tokens")

Encoding simple_wikipedia...
  [simple_wikipedia] 10000 texts, flushed 631,245 tokens
  [simple_wikipedia] 20000 texts, flushed 605,403 tokens
  [simple_wikipedia] 30000 texts, flushed 598,165 tokens
  [simple_wikipedia] 40000 texts, flushed 624,186 tokens
  [simple_wikipedia] 50000 texts, flushed 561,289 tokens
  [simple_wikipedia] 60000 texts, flushed 591,771 tokens
  [simple_wikipedia] 70000 texts, flushed 559,721 tokens
  [simple_wikipedia] 80000 texts, flushed 492,295 tokens
  [simple_wikipedia] 90000 texts, flushed 581,058 tokens
  [simple_wikipedia] 100000 texts, flushed 578,818 tokens
  [simple_wikipedia] 110000 texts, flushed 565,115 tokens
  [simple_wikipedia] 120000 texts, flushed 556,668 tokens
  [simple_wikipedia] 130000 texts, flushed 563,337 tokens
  [simple_wikipedia] 140000 texts, flushed 481,501 tokens
  [simple_wikipedia] 150000 texts, flushed 513,045 tokens
  [simple_wikipedia] 160000 texts, flushed 528,619 tokens
  [simple_wikipedia] 170000 texts, flushed 558,455 t